In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

import sys
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)
    from loguru import logger
    logger.info(f"Project root added to sys.path:\n{PROJECT_ROOT}")
from src.reconstruction.admm_recon import ADMM_Net
from src.utils.fn import *
import torch

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
logger.info(f'Using device: {device}')


In [ ]:
psf = r'D:\qjy\SLMImagingPipeline\output\2025-10-21-exp001\001-psf-EnrF.jpg'
measurement = r"D:\qjy\SLMImagingPipeline\output\2025-10-21-exp001\012-m-18tM.jpg"

save_dir = None              # 设置为 None 的时候不保存
save_name = {
    "gt": "gt.png",
    "psf": "psf.png",
    "meas": "m.png",
    "recon": "r2.png"
}

In [ ]:
downsample = 16

iterations  = 400

mu1_init=1e-4
mu2_init=1e-4
mu3_init=1e-4
tau_init=2.0

ground_truth_file = None                                                        # 设置为 None 的时候不显示

In [ ]:
psf_resized = load_and_downsample_normal2_image(
    psf,
    downsample=downsample,
    mode="gray",
    remove_bg=True,
    normalize=True,
    visualize=False
)
measurement_resized = load_and_downsample_normal2_image(
    measurement,
    downsample=downsample,
    mode="rgb",
    remove_bg=False,
    normalize=True,
    visualize=False
)
measurement_resized = measurement_resized.transpose((2, 0, 1))  # 转为 (C, H, W)

logger.info(f"psf_resized: shape={psf_resized.shape}, dtype={psf_resized.dtype}, min={psf_resized.min():.3f}, max={psf_resized.max():.3f}")
logger.info(f"measurement_resized: shape={measurement_resized.shape}, dtype={measurement_resized.dtype}, min={measurement_resized.min():.3f}, max={measurement_resized.max():.3f}")

input_tensor = (
    torch.from_numpy(measurement_resized.copy())
    .float()
    .unsqueeze(0)
    .to(device)
)

admm_net = ADMM_Net(
    psf_resized,
    iterations=iterations,
    cuda_device=device,
    mu1_init=mu1_init,
    mu2_init=mu2_init,
    mu3_init=mu3_init,
    tau_init=tau_init
)
admm_net.to(device)

admm_net.eval()
with torch.no_grad():
    output_tensor = admm_net(input_tensor)

logger.info(f"input Tensor: shape={input_tensor.shape}, dtype={input_tensor.dtype}, min={input_tensor.min():.3f}, max={input_tensor.max():.3f}")
logger.info(f"output Tensor: shape={output_tensor.shape}, dtype={output_tensor.dtype}, min={output_tensor.min():.3f}, max={output_tensor.max():.3f}")

reconstruction = output_tensor[0].cpu().numpy().transpose(1, 2, 0)


In [ ]:
visualize_reconstruction(
    psf_resized=psf_resized,
    measurement_resized=np.clip(measurement_resized.transpose(1, 2, 0) / measurement_resized.max(), 0, 1),
    reconstruction=reconstruction,
    ground_truth_file=ground_truth_file,
    iterations=iterations
)
save_all_images(
    psf=psf_resized,
    measurement=np.clip(measurement_resized.transpose(1, 2, 0) / measurement_resized.max(), 0, 1),
    reconstruction=reconstruction,
    ground_truth=ground_truth_file,     # 没有真值就自动跳过
    save_dir=save_dir,                  # 保存路径
    filenames=save_name                 # 可选，默认文件名 psf.png, m.png, r.png
)
